In [2]:
import pandas as pd
import numpy as np


In [3]:
#importanção dos dados do arquivo CSV / CSV importing data
df = pd.read_csv('../data/raw/dados_clientes.csv', encoding= 'utf-8')

In [4]:
#correção da coluna de login_flag / correction of the login_flag column
df['login_flag'].value_counts(dropna=False)
df['login_flag'] = df['login_flag'].fillna(0)
df['login_flag'].value_counts()

login_flag
False    472999
True     172001
Name: count, dtype: int64

In [5]:
#soma de login flag por perfil e por mes/ sum of login flag by profile and by month
df.groupby("perfil")['login_flag'].sum()
df.groupby("month")['login_flag'].sum()

month
1     24499
7     24515
8     24673
9     24598
10    24598
11    24548
12    24570
Name: login_flag, dtype: int64

In [6]:
#login_flag por cliente e mês / login_flag by client and month
df.groupby(["cliente_id", "month"])['login_flag'].sum()
#login_flag por perfil e mês / login_flag by profile and month
df.groupby(["perfil", "month"])['login_flag'].sum()

perfil   month
chronic  1         2399
         7         2423
         8         2447
         9         2462
         10        2477
         11        2425
         12        2460
gradual  1        10894
         7        10826
         8        10969
         9        10834
         10       10808
         11       10864
         12       10855
heavy    1         8347
         7         8371
         8         8390
         9         8427
         10        8417
         11        8376
         12        8400
weekend  1         2859
         7         2895
         8         2867
         9         2875
         10        2896
         11        2883
         12        2855
Name: login_flag, dtype: int64

In [7]:
#estatísticas por cliente / statistics by client
df.groupby("cliente_id")['login_flag'].sum().describe()
#frequencia media por mes / avarage frequency by month
df.groupby(["cliente_id", "month"])['login_flag'].mean()

cliente_id  month
1           1        0.419355
            7        0.548387
            8        0.516129
            9        0.500000
            10       0.419355
                       ...   
3000        8        0.032258
            9        0.066667
            10       0.064516
            11       0.066667
            12       0.064516
Name: login_flag, Length: 21000, dtype: float64

In [8]:
#identificando dias consecutivos sem login / identifying consecutive days without login
df = df.sort_values(["cliente_id", "data"])

df["semLogin"] = df["login_flag"] != df.groupby("cliente_id")["login_flag"].shift()
grupo = df.groupby("cliente_id")["semLogin"].cumsum()

df["ordem"] = df.groupby(["cliente_id", grupo]).cumcount()+ 1
df["diasConse"] = df.groupby("cliente_id")["ordem"].max()

print(df["diasConse"].values)
#estatísticas de dias consecutivos sem login / statistics of consecutive days without login
df["diasConse"].describe()

[nan  8. 28. ... nan nan nan]


count    3000.000000
mean       21.202333
std        13.396537
min         4.000000
25%         8.000000
50%        19.000000
75%        33.000000
max        57.000000
Name: diasConse, dtype: float64

In [9]:
#churn flag de dias sem login / churn flag of days without login
base_cliente = df.groupby(["cliente_id", "perfil"])["diasConse"].max().reset_index()
base_cliente["churn_flag"] = np.where(base_cliente["diasConse"] >= 30,  1, 0)

print(base_cliente["churn_flag"].mean() * 100)
print(base_cliente["churn_flag"].value_counts())

0.46666666666666673
churn_flag
0    2986
1      14
Name: count, dtype: int64


In [10]:
#resumo por perfil / sumary by profile
resumo_perfil = base_cliente.groupby("perfil").agg(
    total_cliente = ("cliente_id", "count"),
    cliChurn = ("churn_flag", "sum")
)
resumo_perfil["percentualChurn"] = resumo_perfil["cliChurn"] / resumo_perfil["total_cliente"] * 100
print(resumo_perfil)

         total_cliente  cliChurn  percentualChurn
perfil                                           
chronic           1223         7         0.572363
gradual            725         2         0.275862
heavy              479         2         0.417537
weekend            573         3         0.523560


In [11]:
#análise de sequência de login por dias / analysis of login sequence by days
df = df.sort_values(["cliente_id", "data"])
df["data"] = pd.to_datetime(df["data"])

df["delta"] = df.groupby("cliente_id")["data"].diff().dt.days
df["diferença"] = df["delta"] > 1

df["grupo"] = df.groupby("cliente_id")["diferença"].cumsum()
freq = df.groupby(["cliente_id", "grupo"])["login_flag"].cumcount() + 1

df["frequencia"] = freq.where(df["login_flag"], 0)
print(df[["cliente_id","frequencia", "data", "login_flag"]])


        cliente_id  frequencia       data  login_flag
0                1           1 2025-07-01        True
1                1           2 2025-07-02        True
2                1           0 2025-07-03       False
3                1           4 2025-07-04        True
4                1           5 2025-07-05        True
...            ...         ...        ...         ...
644995        3000           0 2026-01-27       False
644996        3000           0 2026-01-28       False
644997        3000           0 2026-01-29       False
644998        3000           0 2026-01-30       False
644999        3000           0 2026-01-31       False

[645000 rows x 4 columns]


In [12]:
#numero maximo de dias consecutivos por cliente / max number of consecutie days by client
máximo = df.groupby("cliente_id")["frequencia"].max()
print(máximo)
#estatísticas de engajamento por perfil / engagement statistics by profile
engajamento_perfil = df.groupby("perfil")["frequencia"].describe()
print(engajamento_perfil)

cliente_id
1       213
2       204
3       214
4       215
5       214
       ... 
2996    206
2997    213
2998    212
2999    211
3000    205
Name: frequencia, Length: 3000, dtype: int64
            count       mean        std  min  25%   50%    75%    max
perfil                                                               
chronic  262945.0   7.015486  30.903347  0.0  0.0   0.0    0.0  215.0
gradual  155875.0  52.681777  69.163251  0.0  0.0   0.0  105.0  215.0
heavy    102985.0  61.573550  70.966022  0.0  0.0  27.0  121.0  215.0
weekend  123195.0  17.626746  47.046316  0.0  0.0   0.0    0.0  215.0


In [13]:
#análise de variação mês a mês / month - to -month variation analysis
df = df.sort_values(["cliente_id", "data"])
df["data"] = pd.to_datetime(df["data"])
df["ano_mes"] = df["data"].dt.to_period("M")

#soma da frequencia de dias por mes-ano de cada cliente/ sum od frequency of days by month-year of each client
dias_logados_mes = df.groupby(["cliente_id", "ano_mes"])["login_flag"].sum().reset_index()

#max de dias ausentes por mes-ano de cada cliente / max of absent days by month-year of each client
max_ausencia_mes = df.groupby(["cliente_id", "ano_mes"])["diasConse"].max().reset_index()

#merge de ambas as analises / merge of both analysis
analise_mes = pd.merge(max_ausencia_mes, dias_logados_mes, on=["cliente_id", "ano_mes"], how="outer")
print(analise_mes)

#eliminando NANs / eliminating NANs
analise_mes["diasConse"] = analise_mes["diasConse"].fillna(0)
analise_mes["login_flag"] = analise_mes["login_flag"].fillna(0)

#churn flag 
analise_mes["churn_flag"] = np.where(analise_mes["diasConse"] > 15, 1, 0)
#coluna de risco/ risck column
analise_mes["risco"] = np.where(analise_mes["diasConse"] > 15, "Alerta vermelho", "Estado de alerta")

#merge do df original com analise mensal / merge of the original df with monthly analysis
analise_mes = analise_mes.merge(
    df[["cliente_id", "perfil"]].drop_duplicates(),
      on= "cliente_id",
      how="left")

#risco medio / medium risck
risco_medio = analise_mes.groupby("risco")["churn_flag"].mean().reset_index()
print(r"Risco médio: \n ", risco_medio)

# perfil x risco x churn medio / profile x risck x aravage churn
perfil_risco_churn = analise_mes.groupby(["perfil", "risco"])["churn_flag"].count().reset_index()
print("Perfil x Risco x Churn: \n",perfil_risco_churn)

       cliente_id  ano_mes  diasConse  login_flag
0               1  2025-07       47.0          17
1               1  2025-08       49.0          16
2               1  2025-09       48.0          15
3               1  2025-10       48.0          13
4               1  2025-11       49.0          19
...           ...      ...        ...         ...
20995        3000  2025-09        NaN           2
20996        3000  2025-10        NaN           2
20997        3000  2025-11        NaN           2
20998        3000  2025-12        NaN           2
20999        3000  2026-01        NaN           2

[21000 rows x 4 columns]
Risco médio: \n                risco  churn_flag
0   Alerta vermelho         1.0
1  Estado de alerta         0.0
Perfil x Risco x Churn: 
     perfil             risco  churn_flag
0  chronic   Alerta vermelho          49
1  chronic  Estado de alerta        8512
2  gradual   Alerta vermelho          14
3  gradual  Estado de alerta        5061
4    heavy   Alerta vermelho  

In [14]:
analise_mes.to_csv("../data/processed/analise_mensal.csv", index= False)
resumo_perfil.to_csv("../data/processed/resumo_perfil.csv", index=False)